# Cognivision Voice Intelligence
## Notebook 02: Audio Signal Processing & Acoustic EDA
This notebook analyzes raw speech waveforms, Short-Time Fourier Transform (STFT) spectrograms, and Mel spectrograms across different emotion categories in CREMA-D.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from backend.app.preprocessing.audio_pipeline import load_audio, generate_mel_spectrogram
from backend.app.services.dataset_service import dataset_service
from backend.app.utils.constants import EMOTIONS

### 1. Sample Selection Across All 6 Emotions

In [ ]:
df = dataset_service.load_metadata_if_exists()
samples = {}
for emo in EMOTIONS:
    subset = df[df['emotion'] == emo]
    if len(subset) > 0:
        samples[emo] = subset.iloc[0]['file_path']

### 2. Waveform Representations Across Emotions

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, (emo, fpath) in enumerate(samples.items()):
    y, sr = load_audio(fpath, target_sr=16000)
    t = np.linspace(0, len(y) / sr, len(y))
    axes[i].plot(t, y, color='#06b6d4', alpha=0.8)
    axes[i].set_title(f"Emotion: {emo}", fontweight='bold')
    axes[i].grid(alpha=0.25)
    axes[i].set_ylabel("Amplitude")

axes[-1].set_xlabel("Time (s)")
axes[-2].set_xlabel("Time (s)")
plt.suptitle("Speech Waveform Dynamics Across Emotion Categories", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Log-Mel Spectrogram Representation

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 9))
axes = axes.flatten()

for i, (emo, fpath) in enumerate(samples.items()):
    y, sr = load_audio(fpath, target_sr=16000)
    mel = generate_mel_spectrogram(y, sr, n_mels=64)
    img = axes[i].imshow(mel, aspect='auto', origin='lower', cmap='magma')
    axes[i].set_title(f"Log-Mel Spectrogram: {emo}", fontweight='bold')
    axes[i].set_ylabel("Mel Bins")
    fig.colorbar(img, ax=axes[i], format='%+2.0f dB')

axes[-1].set_xlabel("Time Frames")
axes[-2].set_xlabel("Time Frames")
plt.suptitle("Log-Mel Spectrogram Acoustic Energy Across Emotions", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()